In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, MaxNLocator
from adjustText import adjust_text
import statsmodels.api as sm
import mercury as mr
from IPython.display import display, Markdown

plt.rcParams.update({"font.size": 13, "axes.titlesize": 19, "axes.labelsize": 16,
                     "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 11,
                     "figure.dpi": 100, "savefig.dpi": 150})
COLORS = {"Standard": "#0f766e", "Plus": "#0891b2", "Pro": "#2563eb",
          "Pro Max": "#7c3aed", "Budget": "#d97706", "Other": "#9ca3af"}
SCREEN_OPTIONS = {
    "Screen size": ("screen_size_inches", "Screen diagonal (inches)", "inch"),
    "Screen area": ("screen_area_sq_inches", "Screen area (square inches)", "square inch"),
    "Total screen area": ("total_screen_area_sq_inches", "Total screen area (square inches)", "square inch"),
}

def model_line(model):
    # Keep this classification aligned with the article notebook.
    if "SE" in model or model in ["iPhone 5c", "iPhone 16e", "iPhone 17e"]:
        return "Budget"
    if "Pro Max" in model or model == "iPhone XS Max":
        return "Pro Max"
    if "Pro" in model or model in ["iPhone X", "iPhone XS"]:
        return "Pro"
    if "Plus" in model:
        return "Plus"
    if "mini" in model or model in ["iPhone Air", "iPhone Duo"]:
        return "Other"
    return "Standard"

data = pd.read_csv("iphone_prices_adjusted.csv")
data["model_line"] = data.model.map(model_line)

In [ ]:
years = [str(year) for year in range(int(data.release_year.min()), int(data.release_year.max()) + 1)]
year_columns = mr.Columns(2, position="sidebar", min_width="100px", gap="8px", border="", key="year-range")
with year_columns[0]:
    start_year = mr.Select(label="From year", choices=years, value=years[0], position="inline", key="start-year")
with year_columns[1]:
    end_year = mr.Select(label="Through year", choices=years, value="2026" if "2026" in years else years[-1], position="inline", key="end-year")
price_type = mr.Select(label="Prices", choices=["Original price", "Inflation-adjusted price"], value="Inflation-adjusted price")
screen_measure = mr.Select(label="Screen measurement", choices=list(SCREEN_OPTIONS), value="Screen area")
model_lines = mr.MultiSelect(label="Model lines", choices=list(COLORS), value=list(COLORS))
duo_filter = mr.Select(label="iPhone Duo", choices=["Include Duo", "Exclude Duo"], value="Include Duo")
show_fit = mr.Select(label="Trend fits and 95% confidence intervals", choices=["Hide fit", "Show fit"], value="Hide fit")

In [ ]:
def fit_trend(frame, x_column, y_column, logarithmic=False):
    clean = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=[x_column, y_column])
    if logarithmic:
        clean = clean[clean[y_column] > 0]
    x = clean[x_column].to_numpy(dtype=float)
    observed = clean[y_column].to_numpy(dtype=float)
    if len(x) < 3 or np.unique(x).size < 2:
        return None
    target = np.log(observed) if logarithmic else observed
    center = float(x.mean())
    fitted = sm.OLS(target, sm.add_constant(x - center)).fit()
    grid = np.linspace(x.min(), x.max(), 160)
    prediction = fitted.get_prediction(sm.add_constant(grid - center)).summary_frame(alpha=0.05)
    curve = prediction["mean"].to_numpy()
    lower = prediction["mean_ci_lower"].to_numpy()
    upper = prediction["mean_ci_upper"].to_numpy()
    predicted = fitted.fittedvalues
    if logarithmic:
        curve, lower, upper = np.exp(curve), np.exp(lower), np.exp(upper)
        predicted = np.exp(predicted)
    slope_low, slope_high = fitted.conf_int(alpha=0.05)[1]
    return {
        "n": len(x), "center": center, "intercept": float(fitted.params[0]),
        "slope": float(fitted.params[1]), "slope_low": float(slope_low), "slope_high": float(slope_high),
        "r2": float(fitted.rsquared) if np.ptp(target) > 0 else None,
        "rmse": float(np.sqrt(np.mean((observed - predicted) ** 2))),
        "grid": grid, "curve": curve, "lower": lower, "upper": upper,
        "logarithmic": logarithmic,
        "first_year": int(clean.release_year.min()), "last_year": int(clean.release_year.max()),
    }


def selected_label_rows(frame, y_column):
    if len(frame) <= 30:
        return frame
    notable = {"iPhone (1st generation)", "iPhone X", "iPhone XS Max", "iPhone 12",
               "iPhone SE (3rd generation)", "iPhone 17", "iPhone 17 Pro Max", "iPhone Duo"}
    indices = set(frame.index[frame.model.isin(notable)])
    indices.update([frame[y_column].idxmin(), frame[y_column].idxmax()])
    # Include the newest selected member of each family.
    indices.update(frame.sort_values(["release_year", "model"]).groupby("model_line").tail(1).index)
    return frame.loc[sorted(indices)]


def draw_chart(frame, x_column, y_column, x_label, title, y_label, y_top, fit):
    clean = frame.replace([np.inf, -np.inf], np.nan).dropna(subset=[x_column, y_column])
    if clean.empty:
        return None
    fig, ax = plt.subplots(figsize=(14, 8), layout="constrained")
    ax.set_facecolor("#f8fafc")
    for line, color in COLORS.items():
        group = clean[clean.model_line == line]
        if not group.empty:
            ax.scatter(group[x_column], group[y_column], s=80, color=color,
                       edgecolors="white", linewidths=0.9, zorder=3, label=line)
    if fit is not None:
        name = "Log-linear trend" if fit["logarithmic"] else "Linear fit"
        ax.plot(fit["grid"], fit["curve"], color="#12234a", lw=2.5, label=name, zorder=2)
        ax.fill_between(fit["grid"], fit["lower"], fit["upper"], color="#94a3b8",
                        alpha=0.22, label="95% confidence interval", zorder=1)
    ax.set_title(title, pad=18, fontweight="bold")
    ax.set(xlabel=x_label, ylabel=y_label, ylim=(0, y_top))
    ax.yaxis.set_major_formatter(StrMethodFormatter("${x:,.0f}"))
    if x_column == "release_year":
        ax.xaxis.set_major_locator(MaxNLocator(integer=True, nbins=11))
    x = clean[x_column].to_numpy(dtype=float)
    margin = max(np.ptp(x) * 0.12, 1 if x_column == "release_year" else 0.4)
    ax.set_xlim(x.min() - margin, x.max() + margin)
    ax.grid(alpha=0.20)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(loc="upper left", ncol=4, frameon=True, fontsize=10)
    label_rows = selected_label_rows(clean, y_column)
    texts = [ax.text(row[x_column], row[y_column], ("Original iPhone" if row.model == "iPhone (1st generation)" else row.model.removeprefix("iPhone ")),
                     fontsize=13, color="#172554", ha="center") for _, row in label_rows.iterrows()]
    fig.canvas.draw()
    fig.set_layout_engine(None)
    if y_column == "price_per_sq_inch":
        # Start same-year labels at distinct heights before vertical-only adjustment.
        previous_height = {}
        for position in np.argsort(label_rows[y_column].to_numpy()):
            row = label_rows.iloc[position]
            year = row[x_column]
            height = max(row[y_column], previous_height.get(year, -np.inf) + y_top * 0.06)
            texts[position].set_y(height)
            previous_height[year] = height
    np.random.seed(7)
    # Constrain every adjustment phase so year-based area labels stay at their own year.
    movement = {phase: "y" for phase in ("text", "static", "explode", "pull")} if y_column == "price_per_sq_inch" else {phase: "xy" for phase in ("text", "static", "explode", "pull")}
    adjust_text(texts, x=x, y=clean[y_column].to_numpy(), ax=ax, iter_lim=70,
                target_x=label_rows[x_column].to_numpy(), target_y=label_rows[y_column].to_numpy(),
                only_move=movement, prevent_crossings=y_column != "price_per_sq_inch", min_arrow_len=0,
                expand=(1.1, 1.2), force_text=(0.4, 0.6), ensure_inside_axes=True,
                arrowprops=dict(arrowstyle="-", color="#94a3b8", lw=0.8))
    return fig


def fit_summary(fit, title, x_label, unit, y_unit="USD"):
    r2 = f"{fit['r2']:.3f}" if fit['r2'] is not None else "Undefined (constant values)"
    if fit["logarithmic"]:
        change = np.expm1(fit["slope"]) * 100
        low, high = np.expm1([fit["slope_low"], fit["slope_high"]]) * 100
        direction = "decreases" if change < 0 else "increases"
        sentence = f"Within this selection, fitted price per square inch {direction} by about **{abs(change):.1f}% per year**."
        slope_row = f"| Annual change (95% CI) | {change:+.2f}% [{low:+.2f}%, {high:+.2f}%] |"
        equation = f"estimated price per in² = exp({fit['intercept']:.4f} {fit['slope']:+.4f} × (year − {fit['center']:.2f}))"
        r_label = "R² (log prices per in²)"
    else:
        direction = "higher" if fit["slope"] >= 0 else "lower"
        sentence = f"Within this selection, each additional {unit} is associated with about **${abs(fit['slope']):,.0f} {direction} prices**."
        slope_row = (f"| Slope (USD per {unit}) | {fit['slope']:+,.2f} |\n"
                     f"| 95% CI for slope | [{fit['slope_low']:+,.2f}, {fit['slope_high']:+,.2f}] |")
        equation = f"estimated price = {fit['intercept']:,.2f} {fit['slope']:+,.2f} × (x − {fit['center']:.2f}), where x is {x_label.lower()}"
        r_label = "R²"
    display(Markdown(
        f"### Fit summary · {title}\n{sentence}\n\n"
        f"| Statistic | Value |\n| :--- | ---: |\n"
        f"| Models used | {fit['n']} |\n| Fitting period | {fit['first_year']}–{fit['last_year']} |\n"
        f"| {r_label} | {r2} |\n{slope_row}\n| RMSE ({y_unit}) | {fit['rmse']:,.2f} |\n\n"
        f"**Equation:** {equation}."
    ))

In [ ]:
display(Markdown("# 📱 iPhone Price Explorer\nExplore launch prices and screen measurements across iPhone generations."))
display(Markdown("[Code & data ↗](https://github.com/mljar/mercury-examples/tree/main/iphone-dashboard) · [Read the article ↗](https://mljar.com/blog/iphone-prices-analysis/)"))

# Validate before producing any dashboard results.
valid = int(start_year.value) <= int(end_year.value)
if not valid:
    display(Markdown("**Choose a From year no later than Through year.**"))
else:
    filtered = data[
        data.release_year.between(int(start_year.value), int(end_year.value))
        & data.model_line.isin(model_lines.value)
    ].copy()
    if duo_filter.value == "Exclude Duo":
        filtered = filtered[filtered.model != "iPhone Duo"]
    filtered = filtered.sort_values(["release_year", "model"])
    if filtered.empty:
        valid = False
        display(Markdown("**No models match these filters.** Choose at least one model line or widen the year range."))

if valid:
    screen_column, screen_label, screen_slope_unit = SCREEN_OPTIONS[screen_measure.value]
    price_column = "price_usd" if price_type.value == "Original price" else "price_usd_adjusted"
    price_basis = "original" if price_column == "price_usd" else "inflation-adjusted"
    priced = filtered.dropna(subset=[price_column]).copy()
    area_column = "total_screen_area_sq_inches" if screen_measure.value == "Total screen area" else "screen_area_sq_inches"
    area_name = "total screen area" if area_column == "total_screen_area_sq_inches" else "main screen area"
    denominator = filtered[area_column].where(filtered[area_column] > 0)
    filtered["price_per_sq_inch"] = filtered[price_column] / denominator
    # Limits depend on the selection, but never on the current price toggle or fit toggle.
    all_prices = filtered[["price_usd", "price_usd_adjusted"]]
    price_max = all_prices.max().max()
    price_y_top = max(1.0, float(price_max) * 1.30) if pd.notna(price_max) else 1.0
    area_max = all_prices.div(denominator, axis=0).max().max()
    area_y_top = max(1.0, float(area_max) * 1.30) if pd.notna(area_max) else 1.0
    money = lambda value: f"${value:,.0f}" if pd.notna(value) else "—"
    display(mr.Indicator([
        mr.Indicator(len(filtered), label="Models", variant="primary"),
        mr.Indicator(money(priced[price_column].median()), label="Median price", variant="teal"),
        mr.Indicator(money(priced[price_column].min()), label="Lowest price", variant="primary"),
        mr.Indicator(money(priced[price_column].max()), label="Highest price", variant="teal"),
    ]))
    display(Markdown(f"**{price_type.value}** · {len(priced)} of {len(filtered)} models have a recorded price · Snapshot: {data.status_as_of.max()}"))

if valid:
    charts = [
        ("release_year", price_column, "Release year", "Launch prices over time", "year", False, price_y_top),
        (screen_column, price_column, screen_label, f"{screen_measure.value} & price", screen_slope_unit, False, price_y_top),
        ("release_year", "price_per_sq_inch", "Release year", f"Price per square inch · {area_name}", "year", True, area_y_top),
    ]
    fit_results = {}
    for x_col, y_col, x_label, title, unit, logarithmic, y_top in charts:
        fit = fit_trend(filtered, x_col, y_col, logarithmic) if show_fit.value == "Show fit" else None
        fit_results[title] = fit
        y_label = "Price per square inch (USD)" if logarithmic else f"{price_type.value} (USD)"
        figure = draw_chart(filtered, x_col, y_col, x_label, f"{title} · {price_basis}", y_label, y_top, fit)
        if figure is None:
            display(Markdown(f"**{title}:** no models have both a recorded price and the required measurement."))
        else:
            display(figure)
            plt.close(figure)
            if fit is not None:
                fit_summary(fit, title, x_label, unit, "USD/in²" if logarithmic else "USD")
            elif show_fit.value == "Show fit":
                display(Markdown("A fit needs at least three complete observations and two distinct x values; log trends also need positive prices."))
    if "iPhone Duo" in filtered.model.values:
        display(Markdown("**Duo included:** its large screen area can strongly influence the fitted slope. Compare with **Exclude Duo**, especially when using total screen area."))
    display(Markdown("## Selected models"))
    table_columns = {
        "model": "Model", "release_year": "Year", "model_line": "Model line",
        "price_usd": "Original price (USD)", "price_usd_adjusted": "Adjusted price (Aug 2026 USD)",
        "screen_area_sq_inches": "Screen area (in²)", "total_screen_area_sq_inches": "Total screen area (in²)",
    }
    mr.Table(filtered[list(table_columns)].rename(columns=table_columns), page_size=10, search=True, key="selected-models")
    display(Markdown("## About the data and fits"))
    display(Markdown("Inflation-adjusted prices are in August 2026 dollars, calculated with the US CPI-U index from the Bureau of Labor Statistics."))
    display(Markdown(
        "Original prices are nominal full-device launch prices with purchase conditions documented in the source CSV. "
        "All release statuses are eligible, including announced models. Missing values are excluded from the relevant chart and fit. "
        "Screen size is the main diagonal; total area includes both screens for foldables. Price per square inch uses "
        f"**{area_name}**. For more than 30 points, only notable models, price extremes, and the latest model in each family are labeled."
    ))
    display(Markdown(
        "The article fits use **2007–2025**. Dashboard fits use the current year and model selection, including 2026 if selected. "
        "The first two charts use ordinary least squares; their 95% bands describe the mean trend, not individual-phone prediction intervals. "
        "The price-per-area chart fits log(price per in²) against year, matching the article; its band is transformed from log space "
        "and describes the geometric trend, not the arithmetic mean. All fits weight models equally and assume independent errors "
        "with constant variance on the fitted scale. R² measures explained variation on that scale; RMSE measures price error "
        "in the displayed units. Associations do not establish causation."
    ))